> 🌐 **机器翻译提示 / Machine-translation notice**：本笔记本的文字说明由机器翻译生成，尚未经过母语者校对；代码与数字结果保持与越南语原版完全一致。

# 第12单元 — 信度、效度与研究伦理
第12单元（最后一个模块，12/12）专用notebook — 重现两个量表的Cronbach's Alpha、题项-总分相关（item-total correlation）、删除后alpha值（alpha-if-deleted），并与第10单元的探索性因子分析（EFA）联系起来，作为结构效度的证据。

对应SPSS/PSPP操作：`Analyze > Scale > Reliability Analysis`，命令 `RELIABILITY`。

In [1]:
# 请先运行本单元格（Colab：点击▶）。无需额外安装。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟的"240名学生班级"数据集 —— 与第03-08单元使用的
    同一个 seed=2026，与仓库中的 lop_hoc_240.csv/.sav 完全一致
    （非真实数据，但可100%重现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已创建 df：", df.shape)

已创建 df： (240, 14)


## 1. Cronbach's Alpha —— 手工计算，与PSPP交叉核对
🎯 **该方法回答什么问题？** Cronbach's Alpha回答：一个量表中的各题项是否"步调一致"地共同变化 —— 它衡量量表的信度（RELIABILITY），尚未说明该量表是否测量了正确的概念（效度，见第3节）。

In [2]:
def cronbach_alpha(items_df):
    items_df = items_df.dropna()
    k = items_df.shape[1]
    item_vars = items_df.var(axis=0, ddof=1)
    total_var = items_df.sum(axis=1).var(ddof=1)
    alpha = (k/(k-1)) * (1 - item_vars.sum()/total_var)
    return alpha, len(items_df)

a_interest, n1 = cronbach_alpha(df[["h1","h2","h3"]])
a_anxiety, n2 = cronbach_alpha(df[["a1","a2","a3"]])
print(f"Cronbach's alpha（学习兴趣，h1-h3）：{a_interest:.4f}（n={n1}）")
print(f"Cronbach's alpha（焦虑，a1-a3）：{a_anxiety:.4f}（n={n2}）")

Cronbach's alpha（学习兴趣，h1-h3）：0.7133（n=237）
Cronbach's alpha（焦虑，a1-a3）：0.7585（n=240）


#### 📤 真实输出
α（学习兴趣）=0.7133，α（焦虑）=0.7585。✅ 与PSPP（`RELIABILITY`）结果完全一致：0.71与0.76（保留2位小数）。

## 2. 题项-总分相关（item-total correlation）与删除后alpha值（alpha-if-deleted）
🎯 **该方法回答什么问题？** 这两个指标回答：每个单独的题项对整体信度的贡献如何 —— 哪个题项在"拉低"alpha值，删除某个题项是否能改善该量表？

In [3]:
items = df[["h1","h2","h3"]].dropna()
total = items.sum(axis=1)
print("--- 学习兴趣量表 ---")
for col in items.columns:
    rest = total - items[col]
    r = np.corrcoef(items[col], rest)[0,1]
    others = [c for c in items.columns if c != col]
    a_del, _ = cronbach_alpha(items[others])
    print(f"{col}: item-rest r={r:.3f}, alpha-if-deleted={a_del:.4f}")

--- 学习兴趣量表 ---
h1: item-rest r=0.570, alpha-if-deleted=0.5752
h2: item-rest r=0.492, alpha-if-deleted=0.6716
h3: item-rest r=0.535, alpha-if-deleted=0.6195


#### 📤 真实输出
h1：r=0.570，alpha-if-deleted=0.575。h2：r=0.492，alpha-if-deleted=0.672。h3：r=0.535，alpha-if-deleted=0.620。✅ 与PSPP结果一致（0.57/0.58；0.49/0.67；0.54/0.62）。删除任意一个题项都会使alpha低于当前的0.713 —— 保留全部3个题项是最佳选择。

## 3. 结构效度 —— 与第10单元探索性因子分析（EFA）的联系
🎯 **该方法回答什么问题？** 将EFA结果（第10单元）与最初的理论设计进行对照，回答的问题是：该量表是否测量了预期的理论概念（结构效度）—— 这与"内部是否一致"（信度，第1-2节）是完全不同的问题。

In [4]:
from factor_analyzer import FactorAnalyzer
items6 = df[["h1","h2","h3","a1","a2","a3"]].dropna()
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items6)
loadings = pd.DataFrame(fa.loadings_, index=items6.columns, columns=["因子1", "因子2"])
print(loadings.round(3))
print("\n-> h1-h3 明显载荷在同一个因子上，a1-a3 明显载荷在另一个因子上")
print("-> 与最初的理论设计（2个概念：学习兴趣、焦虑）完全吻合")
print("-> 这是支持两个量表结构效度的证据")

      因子1    因子2
h1 -0.043  0.747
h2 -0.003  0.603
h3 -0.096  0.670
a1  0.725 -0.092
a2  0.722 -0.083
a3  0.695 -0.016

-> h1-h3 明显载荷在同一个因子上，a1-a3 明显载荷在另一个因子上
-> 与最初的理论设计（2个概念：学习兴趣、焦虑）完全吻合
-> 这是支持两个量表结构效度的证据


#### 📤 真实输出
EFA结构与最初的设计完全吻合（与第10单元一致）—— 结合第1节中可接受的Cronbach's Alpha，我们现在对这两个量表的信度（alpha）与结构效度（EFA与设计吻合）都有了相当扎实的证据。

## 4. 用模拟数据说明"高信度、低效度"
🎯 经典的"秤偏差2公斤"例子：构造一个带有固定系统性误差的测量，说明即使测量本身不准确，信度（alpha）仍然可以很高。

In [5]:
rng = np.random.default_rng(2026)
true_weight = rng.normal(60, 10, 240)  # 真实体重
# "有偏差的秤"：始终多读出2公斤，再加上少量测量噪声（用同一台有偏差的秤测量3次）
bias = 2.0
measured_1 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_2 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_3 = true_weight + bias + rng.normal(0, 0.3, 240)

fake_scale = pd.DataFrame({"lan1": measured_1, "lan2": measured_2, "lan3": measured_3})
alpha_fake, _ = cronbach_alpha(fake_scale)
mean_error = (fake_scale.mean(axis=1) - true_weight).mean()
print(f"'有偏差的秤'（3次测量）的Cronbach's alpha：{alpha_fake:.4f}  <- 非常高（信度好）")
print(f"与真实体重相比的平均误差：{mean_error:+.2f} kg  <- 始终存在偏差（效度差）")

'有偏差的秤'（3次测量）的Cronbach's alpha：0.9997  <- 非常高（信度好）
与真实体重相比的平均误差：+2.00 kg  <- 始终存在偏差（效度差）


#### 📤 真实输出
"有偏差的秤"的alpha非常高（>0.95 —— 3次读数几乎完全相同），但与真实体重相比的平均误差始终是+2.00kg。用模拟数据证明：**高信度并不能保证高效度** —— 这正是第12单元的核心结论。